# 04 — Feature Engineering

## Purpose
Create meaningful fraud signals that will serve as node features in the graph.

> "What information helps detect fraud?"

### Feature Categories:
1. **Transaction Features** — amount, timing, velocity
2. **Identity Features** — device patterns, email patterns, card behaviour
3. **Risk Features** — relational signals that connect entities
4. **Aggregated Features** — groupby statistics per entity

In [1]:
# =============================================================================
# 1. Environment Setup
# =============================================================================
import warnings
warnings.filterwarnings('ignore')
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path(os.getcwd()).parent if 'notebooks' in str(os.getcwd()) else Path(os.getcwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import seaborn as sns
print('Imports complete.')


Imports complete.


# 2. Load Processed Data

Start from the output of Notebook 03.

In [2]:
# Load processed data from Notebook 03
processed_path = PROJECT_ROOT / 'data' / 'processed' / 'processed_fraud_data.parquet'

print(f'Looking for: {processed_path}')
print(f'File exists: {processed_path.exists()}')

# In production: df = pd.read_parquet(processed_path)
# For now, show the feature engineering pipeline


Looking for: /Users/airm2/Desktop/My ML Material/graph-fraud-ai/data/processed/processed_fraud_data.parquet
File exists: False


# 3. Transaction Features

Features derived from the transaction itself.

In [3]:
# Transaction Features
print('=' * 70)
print('TRANSACTION FEATURES')
print('=' * 70)
print()

txn_features = {
    'TransactionAmt_log': 'Log transform of transaction amount (handles skew)',
    'TransactionDT_normalized': 'Time since first transaction [0, 1]',
    'TransactionAmt_per_hour': 'Hour-of-day from TransactionDT',
    'TransactionAmt_is_weekend': 'Binary: transaction on weekend',
    'TransactionAmt_zscore': 'Z-score of amount (outlier detection)',
}

for feat, desc in txn_features.items():
    print(f'  • {feat}: {desc}')


TRANSACTION FEATURES

  • TransactionAmt_log: Log transform of transaction amount (handles skew)
  • TransactionDT_normalized: Time since first transaction [0, 1]
  • TransactionAmt_per_hour: Hour-of-day from TransactionDT
  • TransactionAmt_is_weekend: Binary: transaction on weekend
  • TransactionAmt_zscore: Z-score of amount (outlier detection)


In [4]:
# Implementation: Transaction features
print()
print('Implementation:')
print()
# df['TransactionAmt_log'] = np.log1p(df['TransactionAmt'])
# df['TransactionAmt_zscore'] = (df['TransactionAmt'] - df['TransactionAmt'].mean()) / df['TransactionAmt'].std()
# df['TransactionAmt_is_weekend'] = (df['TransactionDT'] % 604800 >= 345600).astype(int)

print('  df["TransactionAmt_log"] = np.log1p(df["TransactionAmt"])')
print('  df["TransactionAmt_zscore"] = zscore(df["TransactionAmt"])')
print('  df["TransactionAmt_is_weekend"] = (df["TransactionDT"] % 604800 >= 345600).astype(int)')



Implementation:

  df["TransactionAmt_log"] = np.log1p(df["TransactionAmt"])
  df["TransactionAmt_zscore"] = zscore(df["TransactionAmt"])
  df["TransactionAmt_is_weekend"] = (df["TransactionDT"] % 604800 >= 345600).astype(int)


# 4. Identity Features

Features from device, email, and card behaviour.

In [5]:
# Identity Features
print('=' * 70)
print('IDENTITY FEATURES')
print('=' * 70)
print()

identity_features = {
    'device_fingerprint_hash': 'Hashed device identifier for graph nodes',
    'email_domain_encoded': 'Numerical email domain encoding',
    'card_is_new': 'Binary: first use of this card in dataset',
    'id_30_browser_type': 'Browser type from id_30',
    'id_30_os_type': 'OS type from id_30',
}

for feat, desc in identity_features.items():
    print(f'  • {feat}: {desc}')


IDENTITY FEATURES

  • device_fingerprint_hash: Hashed device identifier for graph nodes
  • email_domain_encoded: Numerical email domain encoding
  • card_is_new: Binary: first use of this card in dataset
  • id_30_browser_type: Browser type from id_30
  • id_30_os_type: OS type from id_30


# 5. Risk Features (Relational Signals)

These features capture relationships between entities — the foundation for graph construction.

In [6]:
# Risk Features - Relational Signals
print('=' * 70)
print('RISK FEATURES (RELATIONAL SIGNALS)')
print('=' * 70)
print()

risk_features = {
    'num_transactions_per_card': 'Count of transactions per card1',
    'num_fraud_per_card': 'Count of fraud per card1',
    'fraud_rate_per_card': 'Fraud rate for this card (target encoding)',
    'num_transactions_per_device': 'Count per DeviceInfo',
    'num_transactions_per_email': 'Count per P_emaildomain',
    'num_cards_per_email': 'Unique cards per email domain',
    'num_transactions_per_addr1': 'Count per addr1',
    'avg_amount_per_card': 'Mean transaction amount per card',
    'std_amount_per_card': 'Std transaction amount per card',
    'max_amount_per_card': 'Max transaction amount per card',
    'amount_deviation_from_card_mean': 'How much this txn deviates from card avg',
}

for feat, desc in risk_features.items():
    print(f'  • {feat}: {desc}')


RISK FEATURES (RELATIONAL SIGNALS)

  • num_transactions_per_card: Count of transactions per card1
  • num_fraud_per_card: Count of fraud per card1
  • fraud_rate_per_card: Fraud rate for this card (target encoding)
  • num_transactions_per_device: Count per DeviceInfo
  • num_transactions_per_email: Count per P_emaildomain
  • num_cards_per_email: Unique cards per email domain
  • num_transactions_per_addr1: Count per addr1
  • avg_amount_per_card: Mean transaction amount per card
  • std_amount_per_card: Std transaction amount per card
  • max_amount_per_card: Max transaction amount per card
  • amount_deviation_from_card_mean: How much this txn deviates from card avg


In [7]:
# Implementation: Risk features via groupby
print()
print('Implementation (groupby aggregations):')
print()

# card_agg = df.groupby('card1')['isFraud'].agg(['count', 'sum', 'mean'])
# df['num_transactions_per_card'] = df['card1'].map(card_agg['count'])
# df['fraud_rate_per_card'] = df['card1'].map(card_agg['mean'])
# df['avg_amount_per_card'] = df.groupby('card1')['TransactionAmt'].transform('mean')
# df['amount_deviation'] = df['TransactionAmt'] - df['avg_amount_per_card']

print('  card_agg = df.groupby("card1")["isFraud"].agg(["count", "sum", "mean"])')
print('  df["fraud_rate_per_card"] = df["card1"].map(card_agg["mean"])')
print('  df["num_transactions_per_card"] = df["card1"].map(card_agg["count"])')
print('  df["amount_deviation"] = df["TransactionAmt"] - df.groupby("card1")["TransactionAmt"].transform("mean")')

print()
print('These aggregations create the "relational context" that GNNs will', )
print('learn to exploit through message passing.')



Implementation (groupby aggregations):

  card_agg = df.groupby("card1")["isFraud"].agg(["count", "sum", "mean"])
  df["fraud_rate_per_card"] = df["card1"].map(card_agg["mean"])
  df["num_transactions_per_card"] = df["card1"].map(card_agg["count"])
  df["amount_deviation"] = df["TransactionAmt"] - df.groupby("card1")["TransactionAmt"].transform("mean")

These aggregations create the "relational context" that GNNs will
learn to exploit through message passing.


# 6. Save Features

Output the final feature matrix.

In [8]:
# Save final features
print('=' * 70)
print('OUTPUT')
print('=' * 70)
print()

features_dir = PROJECT_ROOT / 'data' / 'processed'
features_path = features_dir / 'final_node_features.npy'
features_dir.mkdir(parents=True, exist_ok=True)

# In production: np.save(features_path, feature_matrix)
# print(f'Saved {feature_matrix.shape} features to {features_path}')

print(f'Output location: {features_path}')
print('Expected output: final_node_features.npy')
print()
print('These features become the node attributes in the graph.')
print()
print('→ Next: Notebook 05 builds the graph structure (edges) on top of these features.')


OUTPUT

Output location: /Users/airm2/Desktop/My ML Material/graph-fraud-ai/data/processed/final_node_features.npy
Expected output: final_node_features.npy

These features become the node attributes in the graph.

→ Next: Notebook 05 builds the graph structure (edges) on top of these features.


# Summary

| Category | Features | Purpose |
|----------|----------|----------|
| Transaction | 5 features | Amount, timing, outlier signals |
| Identity | 5 features | Device, email, card context |
| Risk (Relational) | 11 features | Graph-relevant aggregations |
| **Total** | **21+ features** | Node attributes for GNN |

> **Key Insight:** The risk features (fraud_rate_per_card, num_transactions_per_device, etc.) encode the relational information that traditional ML ignores but GNNs naturally exploit.